# NB00 — Synthetic Equity Market Engine

**Day 2 | Act I: Build the World**  
**Course:** *BUILD THE MACHINE — From a Synthetic Market to an Autonomous Investment Institution*  
**Safety boundary:** synthetic research and education only; no live market data, credentials, brokerage connection, orders, capital allocation, or external execution authority.

## Introduction: constructing the world before asking intelligence to learn

NB00 creates the canonical market world on which the rest of the course will operate. That ordering is deliberate. A model cannot be evaluated responsibly when the origin, structure, defects, and permissible use of its data are unclear. An agent cannot reason reliably when each notebook silently reconstructs a different market history. A backtest cannot be audited when prices, regimes, liquidity, and corporate actions are detached from provenance. The Synthetic Equity Market Engine therefore precedes the first predictive model. Its purpose is to give every later experiment a shared, reproducible, inspectable environment whose behavior is complex enough to teach financial reasoning but controlled enough to support exact validation.

The notebook follows NB00A, the Autonomous Systems Protocol. NB00A defined how tools, skills, agents, shared state, provenance, approval, audit events, and registry entries must be represented. NB00 applies that constitutional language to the data layer. It does not merely generate a table and save it. It separates generation, persistence, and validation into narrow capabilities; records their permissions and tests; produces manifests and hashes; creates known defects; and registers the resulting tools. This is the first full example of the course’s governing principle: a computational capability becomes institutional knowledge only when its source, behavior, limitations, and authority boundary travel with it.

Why begin with synthetic data instead of historical prices? Historical data are indispensable later, but they conceal the true data-generating process. We observe what happened without knowing exactly why it happened, which regimes were latent, which vendor corrections were applied, or which apparent anomalies are real. A synthetic world gives us controlled ground truth. We select the seed, regime transition process, factor structure, number of assets, time horizon, liquidity behavior, and injected defects. We can reproduce the same world, deliberately perturb it, and test whether models and validators detect properties that are known by construction. Synthetic data do not prove investment performance; they provide a governed laboratory for learning whether the research system itself behaves correctly.

The canonical configuration contains thirty equities divided evenly across six sectors—Technology, Financials, Healthcare, Industrials, Consumer, and Energy—and 756 business days beginning on 2 January 2023. The engine assigns every security a market beta, sector exposure, and idiosyncratic volatility. A persistent transition matrix moves the market through four states: calm, trend, mean-reverting, and crisis. Each state changes the drift and volatility of the market factor; the mean-reverting state responds against the preceding return; and crisis periods increase negative drift, sector shock dispersion, volume, and spreads. The resulting environment contains structured nonstationarity rather than a single Gaussian return stream.

Prices are generated from the cumulative return process and expanded into internally consistent open, high, low, close, and adjusted-close fields. Overnight variation determines the opening price, while a positive intraday range creates highs above and lows below the open and close. Volume varies across securities and expands in trend and crisis regimes. Bid–ask spreads are positive and widen sharply during crisis. These choices are pedagogical: future strategies must confront changing volatility, liquidity, and cross-sectional structure instead of operating in a frictionless world. The notebook also creates dividend events and two explicit stock splits. Those events are recorded as teaching inputs, while the manifest plainly states that corporate actions are not fully reflected in retroactively adjusted histories.

A controlled defect layer is created separately from the clean canonical data. It contains a missing close, negative volume, an inconsistent high below the low, a duplicated instrument-date key, and a stale-price run. This separation is crucial. We do not contaminate the research database to make an exercise interesting; we preserve a clean source of truth and maintain a defective copy whose errors are intentional and documented. The dataset validator must accept the clean panel and reject the corrupted panel for the correct reasons. A validator that merely passes clean data has not demonstrated much. Negative controls show that the quality gate can stop a pipeline when important financial invariants are violated.

The persistence layer is a constrained SQLite database rather than a collection of unrelated CSV files. Tables represent instruments, the trading calendar, market regimes, regime history, daily prices, corporate actions, known defects, the dataset manifest, and provenance. Primary keys prevent duplicate identifiers, foreign keys preserve relationships, and check constraints enforce positive prices, non-negative volume and spreads, and coherent OHLC ranges. The build process finishes with SQLite integrity and foreign-key checks plus row-count reconciliation. CSV exports remain useful for inspection, but the database becomes the authoritative substrate consumed by later notebooks.

Reproducibility is strengthened by a deterministic random seed and content hashes for the principal tables. The manifest records dataset identity, version, seed, asset and date counts, horizon, scope, currency, hashes, and limitations. Hashes allow downstream notebooks to determine whether they are analyzing the expected bytes rather than a similarly named file. Provenance records identify the producing capability, source dataset, code version, parameters, and content hash. This turns data lineage from an explanatory paragraph into a machine-verifiable relationship.

NB00 concludes by registering three reusable tools. The Synthetic Equity Generator creates the controlled market bundle. The Canonical Database Builder persists validated tables under relational constraints. The Dataset Validator checks schema completeness, nulls, duplicate keys, price positivity, OHLC consistency, volume, spreads, stale observations, database integrity, and foreign keys. Each tool declares inputs, outputs, errors, permissions, side effects, provenance, tests, and status. None receives brokerage or live-market authority. Later notebooks call these capabilities instead of recreating the market independently.

Within the full course, NB00 completes the data foundation of **Build the World**. NB00A supplies the constitution; NB00 supplies the governed environment; NB00B uses both to prove the first traceable KNN vertical slice. NB01 then compares models, strategies, and portfolio rules. Subsequent acts package tools into skills, assign skills to bounded agents, coordinate research and control agents, and ultimately construct a meta-agent. At every stage, the market substrate remains reproducible and its limitations remain visible. The machine is not asked to learn from an unexplained world. It learns inside a world whose assumptions, defects, invariants, and evidence were designed before prediction began.


## Workflow and course handoff

| Stage | Research question | Notebook action | Governed output |
|---|---|---|---|
| 1. Initialize | Where will the market artifacts live? | Creates a bounded runtime workspace and records the environment | Reproducible local structure |
| 2. Export capabilities | Which operations must remain separate? | Writes generator, database builder, and validator modules | Three narrow implementations |
| 3. Generate | What market world will later models observe? | Simulates assets, regimes, factors, OHLCV, spreads, and actions | Clean canonical bundle and manifest |
| 4. Challenge quality | Can the controls recognize known failures? | Tests clean and deliberately defective panels | Positive and negative validation evidence |
| 5. Persist | How are identities, relationships, and financial invariants enforced? | Builds constrained SQLite tables | Integrity-checked canonical database |
| 6. Diagnose | Does the world contain the intended sector and regime variation? | Reconciles counts, dates, sectors, and regimes | Market summary |
| 7. Register | Can later notebooks reuse the capabilities safely? | Writes ToolSpecs, hashes, and registry entries | Governed tool registry |
| 8. Hand forward | What does NB00B receive? | Publishes data, database, tools, and evidence | Stable substrate for the KNN vertical slice |


## 1. Initialize the bounded synthetic-market workspace

The notebook creates a local workspace for datasets, the database, tools, the registry, and reports, then records the Python runtime and deterministic seed. This explicit structure separates data, executable logic, control evidence, and registration artifacts. No live data source or external system is contacted.


In [ ]:
from pathlib import Path
import json, os, platform, sys
BASE=Path('/content') if Path('/content').exists() and os.access('/content',os.W_OK) else Path('/tmp')
ROOT=BASE/'nb00_synthetic_equity';
for p in ['tools','datasets','database','reports','figures','registry']: (ROOT/p).mkdir(parents=True,exist_ok=True)
print({'python':platform.python_version(),'workspace':str(ROOT),'seed':42})


## 2. Export the three Day 2 tools

Generation, persistence, and validation are intentionally separate. The notebook writes each implementation into the runtime workspace and imports it through the same module boundary that downstream notebooks can use. This prevents the notebook interface from diverging from the reusable capability and prepares the functions for governed ToolSpecs and registration.


In [ ]:
GENERATOR_SOURCE='from __future__ import annotations\nimport hashlib\nimport json\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nSECTORS = ["Technology", "Financials", "Healthcare", "Industrials", "Consumer", "Energy"]\nREGIMES = ["calm", "trend", "mean_revert", "crisis"]\n\n@dataclass(frozen=True)\nclass SyntheticConfig:\n    seed: int = 42\n    n_assets: int = 30\n    n_days: int = 756\n    start: str = "2023-01-02"\n    base_currency: str = "USD"\n\ndef _sha256_frame(df: pd.DataFrame) -> str:\n    return hashlib.sha256(df.to_csv(index=False, float_format="%.10f").encode()).hexdigest()\n\ndef generate_synthetic_equity_market(config: SyntheticConfig = SyntheticConfig()) -> dict:\n    if config.n_assets % len(SECTORS):\n        raise ValueError("n_assets must be divisible by six sectors")\n    rng = np.random.default_rng(config.seed)\n    dates = pd.bdate_range(config.start, periods=config.n_days)\n    tickers = [f"SYN{i:02d}" for i in range(1, config.n_assets + 1)]\n    sector = np.repeat(SECTORS, config.n_assets // len(SECTORS))\n    instruments = pd.DataFrame({\n        "instrument_id": np.arange(1, config.n_assets + 1), "ticker": tickers,\n        "name": [f"Synthetic {s} Company {i+1}" for i, s in enumerate(sector)],\n        "asset_class": "equity", "sector": sector, "currency": config.base_currency,\n        "exchange": "SYNX", "active": 1,\n    })\n\n    transition = np.array([[.94,.035,.02,.005],[.06,.89,.035,.015],[.06,.05,.86,.03],[.18,.05,.07,.70]])\n    regime_idx = np.zeros(config.n_days, dtype=int)\n    for t in range(1, config.n_days): regime_idx[t] = rng.choice(4, p=transition[regime_idx[t-1]])\n    regime_params = {\n        0: (0.00025, 0.007), 1: (0.00075, 0.010), 2: (0.00005, 0.012), 3: (-0.0025, 0.030)\n    }\n    market = np.array([rng.normal(*regime_params[r]) for r in regime_idx])\n    # Mean-reversion days pull against the previous market return.\n    mr = regime_idx == 2\n    market[mr] -= .35 * np.r_[0.0, market[:-1]][mr]\n    sector_shocks = rng.normal(0, 0.0045, size=(config.n_days, len(SECTORS)))\n    crisis = regime_idx == 3\n    sector_shocks[crisis] += rng.normal(-0.0005, 0.009, size=(crisis.sum(), len(SECTORS)))\n    betas = rng.uniform(.75, 1.25, config.n_assets)\n    sector_betas = rng.uniform(.55, .95, config.n_assets)\n    idio_vol = rng.uniform(.006, .014, config.n_assets)\n    returns = np.zeros((config.n_days, config.n_assets))\n    for j in range(config.n_assets):\n        s = SECTORS.index(sector[j])\n        returns[:,j] = betas[j]*market + sector_betas[j]*sector_shocks[:,s] + rng.normal(0,idio_vol[j],config.n_days)\n    returns = np.clip(returns, -.25, .25)\n    start_prices = rng.uniform(25, 180, config.n_assets)\n    close = start_prices * np.exp(np.cumsum(returns, axis=0))\n    overnight = rng.normal(0, .0025, size=close.shape)\n    open_ = np.vstack([start_prices, close[:-1]]) * np.exp(overnight)\n    intraday = np.abs(rng.normal(.006, .003, size=close.shape))\n    high = np.maximum(open_, close) * (1 + intraday)\n    low = np.minimum(open_, close) * np.maximum(.01, 1 - intraday)\n    adj_close = close.copy()\n    base_volume = rng.lognormal(13.3, .45, config.n_assets)\n    vol_multiplier = np.where(regime_idx==3, 2.3, np.where(regime_idx==1, 1.25, 1.0))\n    volume = np.maximum(100, (base_volume[None,:] * vol_multiplier[:,None] * rng.lognormal(0,.25,close.shape)).astype(int))\n    spread_bps = np.maximum(1.0, rng.normal(7, 2, close.shape) * np.where(regime_idx[:,None]==3,2.5,1.0))\n\n    rows=[]\n    for t, date in enumerate(dates):\n        for j in range(config.n_assets):\n            rows.append((j+1,date.date().isoformat(),float(open_[t,j]),float(high[t,j]),float(low[t,j]),float(close[t,j]),float(adj_close[t,j]),int(volume[t,j]),float(spread_bps[t,j]),int(regime_idx[t])))\n    prices = pd.DataFrame(rows, columns=["instrument_id","date","open","high","low","close","adj_close","volume","spread_bps","regime_id"])\n    regimes = pd.DataFrame({"date": dates.date.astype(str), "regime_id": regime_idx, "regime": [REGIMES[i] for i in regime_idx], "market_return": market})\n    calendar = pd.DataFrame({"date": dates.date.astype(str), "is_trading_day": 1, "session": "regular"})\n\n    actions=[]\n    for j in range(config.n_assets):\n        for idx in [190, 440, 690]:\n            if idx < config.n_days:\n                actions.append((j+1, dates[idx].date().isoformat(), "dividend", round(float(rng.uniform(.05,.75)),4), 1.0))\n    # Two explicit split events. Prices are not retroactively adjusted here; the event is a teaching input.\n    for j, idx in [(4,500),(19,620)]:\n        if idx < config.n_days: actions.append((j+1, dates[idx].date().isoformat(), "split", 0.0, 2.0))\n    corporate_actions = pd.DataFrame(actions, columns=["instrument_id","date","action_type","cash_amount","split_ratio"])\n\n    defects=[]; defective=prices.copy()\n    candidates=[("missing_close", 125, "close", np.nan), ("negative_volume", 2400, "volume", -1000),\n                ("high_below_low", 6000, "high", None), ("duplicate_row", 9000, None, None),\n                ("stale_price", 12000, "close", None)]\n    for defect_id,(kind,row,col,value) in enumerate(candidates,1):\n        row=min(row,len(defective)-2)\n        if kind=="high_below_low": defective.loc[row,"high"] = defective.loc[row,"low"]*.98\n        elif kind=="duplicate_row": defective = pd.concat([defective, defective.iloc[[row]]], ignore_index=True)\n        elif kind=="stale_price":\n            iid=defective.loc[row,"instrument_id"]; inds=defective.index[defective.instrument_id==iid][300:308]\n            defective.loc[inds,"close"] = defective.loc[inds[0],"close"]\n        else: defective.loc[row,col]=value\n        defects.append((defect_id,kind,int(row),str(col or "row"),"deliberate test defect"))\n    defect_log = pd.DataFrame(defects, columns=["defect_id","defect_type","row_reference","field","purpose"])\n\n    manifest = {\n        "dataset_id":"synthetic_equity_market_v1", "version":"1.0.0", "seed":config.seed,\n        "n_assets":config.n_assets, "n_days":config.n_days, "start":str(dates[0].date()), "end":str(dates[-1].date()),\n        "scope":"synthetic equities across six sectors", "currency":config.base_currency,\n        "hashes":{"instruments":_sha256_frame(instruments),"prices":_sha256_frame(prices),"regimes":_sha256_frame(regimes),"corporate_actions":_sha256_frame(corporate_actions)},\n        "limitations":["Synthetic data are not evidence of investable performance.","Corporate actions are events for teaching and are not fully reflected in adjusted histories.","No live data or brokerage connection."],\n    }\n    return {"instruments":instruments,"calendar":calendar,"regimes":regimes,"prices":prices,"corporate_actions":corporate_actions,"defective_prices":defective,"defect_log":defect_log,"manifest":manifest}\n\ndef save_datasets(bundle: dict, output_dir: str | Path) -> dict:\n    out=Path(output_dir); out.mkdir(parents=True,exist_ok=True); paths={}\n    for name, value in bundle.items():\n        path=out/f"{name}.json" if name=="manifest" else out/f"{name}.csv"\n        if name=="manifest": path.write_text(json.dumps(value,indent=2)+"\\n")\n        else: value.to_csv(path,index=False)\n        paths[name]=str(path)\n    return paths\n'
DATABASE_SOURCE='from __future__ import annotations\nimport json\nimport sqlite3\nfrom pathlib import Path\nimport pandas as pd\n\nSCHEMA_SQL = """\nPRAGMA foreign_keys=ON;\nCREATE TABLE IF NOT EXISTS instruments(instrument_id INTEGER PRIMARY KEY,ticker TEXT UNIQUE NOT NULL,name TEXT NOT NULL,asset_class TEXT NOT NULL CHECK(asset_class=\'equity\'),sector TEXT NOT NULL,currency TEXT NOT NULL,exchange TEXT NOT NULL,active INTEGER NOT NULL CHECK(active IN (0,1)));\nCREATE TABLE IF NOT EXISTS trading_calendar(date TEXT PRIMARY KEY,is_trading_day INTEGER NOT NULL,session TEXT NOT NULL);\nCREATE TABLE IF NOT EXISTS market_regimes(regime_id INTEGER PRIMARY KEY,regime TEXT UNIQUE NOT NULL,description TEXT NOT NULL);\nCREATE TABLE IF NOT EXISTS regime_history(date TEXT PRIMARY KEY,regime_id INTEGER NOT NULL,market_return REAL NOT NULL,FOREIGN KEY(regime_id) REFERENCES market_regimes(regime_id));\nCREATE TABLE IF NOT EXISTS prices_daily(instrument_id INTEGER NOT NULL,date TEXT NOT NULL,open REAL NOT NULL CHECK(open>0),high REAL NOT NULL CHECK(high>0),low REAL NOT NULL CHECK(low>0),close REAL NOT NULL CHECK(close>0),adj_close REAL NOT NULL CHECK(adj_close>0),volume INTEGER NOT NULL CHECK(volume>=0),spread_bps REAL NOT NULL CHECK(spread_bps>=0),regime_id INTEGER NOT NULL,PRIMARY KEY(instrument_id,date),FOREIGN KEY(instrument_id) REFERENCES instruments(instrument_id),FOREIGN KEY(date) REFERENCES trading_calendar(date),FOREIGN KEY(regime_id) REFERENCES market_regimes(regime_id),CHECK(high>=low),CHECK(high>=open),CHECK(high>=close),CHECK(low<=open),CHECK(low<=close));\nCREATE TABLE IF NOT EXISTS corporate_actions(action_id INTEGER PRIMARY KEY AUTOINCREMENT,instrument_id INTEGER NOT NULL,date TEXT NOT NULL,action_type TEXT NOT NULL CHECK(action_type IN (\'dividend\',\'split\')),cash_amount REAL NOT NULL,split_ratio REAL NOT NULL,FOREIGN KEY(instrument_id) REFERENCES instruments(instrument_id));\nCREATE TABLE IF NOT EXISTS data_defects(defect_id INTEGER PRIMARY KEY,defect_type TEXT NOT NULL,row_reference INTEGER NOT NULL,field TEXT NOT NULL,purpose TEXT NOT NULL);\nCREATE TABLE IF NOT EXISTS dataset_manifest(dataset_id TEXT PRIMARY KEY,version TEXT NOT NULL,seed INTEGER NOT NULL,n_assets INTEGER NOT NULL,n_days INTEGER NOT NULL,start_date TEXT NOT NULL,end_date TEXT NOT NULL,scope TEXT NOT NULL,currency TEXT NOT NULL,manifest_json TEXT NOT NULL);\nCREATE TABLE IF NOT EXISTS provenance(artifact_id TEXT PRIMARY KEY,producer TEXT NOT NULL,source_ids TEXT NOT NULL,code_version TEXT NOT NULL,parameters_json TEXT NOT NULL,content_hash TEXT NOT NULL);\nCREATE INDEX IF NOT EXISTS idx_prices_date ON prices_daily(date);\nCREATE INDEX IF NOT EXISTS idx_prices_regime ON prices_daily(regime_id);\nCREATE INDEX IF NOT EXISTS idx_instruments_sector ON instruments(sector);\n"""\n\ndef build_database(bundle: dict, db_path: str | Path) -> dict:\n    path=Path(db_path); path.parent.mkdir(parents=True,exist_ok=True)\n    if path.exists(): path.unlink()\n    con=sqlite3.connect(path)\n    try:\n        con.executescript(SCHEMA_SQL)\n        bundle[\'instruments\'].to_sql(\'instruments\',con,if_exists=\'append\',index=False)\n        bundle[\'calendar\'].to_sql(\'trading_calendar\',con,if_exists=\'append\',index=False)\n        pd.DataFrame({"regime_id":[0,1,2,3],"regime":["calm","trend","mean_revert","crisis"],"description":["low volatility","positive directional persistence","negative serial dependence","high volatility and negative drift"]}).to_sql(\'market_regimes\',con,if_exists=\'append\',index=False)\n        bundle[\'regimes\'][[\'date\',\'regime_id\',\'market_return\']].to_sql(\'regime_history\',con,if_exists=\'append\',index=False)\n        bundle[\'prices\'].to_sql(\'prices_daily\',con,if_exists=\'append\',index=False)\n        bundle[\'corporate_actions\'].to_sql(\'corporate_actions\',con,if_exists=\'append\',index=False)\n        bundle[\'defect_log\'].to_sql(\'data_defects\',con,if_exists=\'append\',index=False)\n        m=bundle[\'manifest\']\n        con.execute("INSERT INTO dataset_manifest VALUES (?,?,?,?,?,?,?,?,?,?)",(m[\'dataset_id\'],m[\'version\'],m[\'seed\'],m[\'n_assets\'],m[\'n_days\'],m[\'start\'],m[\'end\'],m[\'scope\'],m[\'currency\'],json.dumps(m,sort_keys=True)))\n        for name,h in m[\'hashes\'].items(): con.execute("INSERT INTO provenance VALUES (?,?,?,?,?,?)",(f"{m[\'dataset_id\']}_{name}","synthetic_equity_generator",m[\'dataset_id\'],"1.0.0",json.dumps({"seed":m[\'seed\']}),h))\n        con.commit()\n        integrity=con.execute("PRAGMA integrity_check").fetchone()[0]\n        counts={t:con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in [\'instruments\',\'trading_calendar\',\'prices_daily\',\'corporate_actions\',\'data_defects\',\'dataset_manifest\',\'provenance\']}\n        return {"database":str(path),"integrity":integrity,"counts":counts}\n    finally: con.close()\n'
VALIDATOR_SOURCE='from __future__ import annotations\nimport sqlite3\nfrom pathlib import Path\nimport pandas as pd\n\ndef validate_prices(df: pd.DataFrame) -> dict:\n    required=[\'instrument_id\',\'date\',\'open\',\'high\',\'low\',\'close\',\'adj_close\',\'volume\',\'spread_bps\',\'regime_id\']\n    findings=[]\n    missing=[c for c in required if c not in df.columns]\n    if missing: findings.append({"code":"MISSING_COLUMNS","count":len(missing),"detail":missing})\n    if missing: return {"valid":False,"findings":findings}\n    checks={\n      "NULL_REQUIRED":df[required].isna().any(axis=1),\n      "DUPLICATE_KEY":df.duplicated([\'instrument_id\',\'date\'],keep=False),\n      "NONPOSITIVE_PRICE":(df[[\'open\',\'high\',\'low\',\'close\',\'adj_close\']]<=0).any(axis=1),\n      "OHLC_INCONSISTENT":(df.high<df.low)|(df.high<df.open)|(df.high<df.close)|(df.low>df.open)|(df.low>df.close),\n      "NEGATIVE_VOLUME":df.volume<0,\n      "NEGATIVE_SPREAD":df.spread_bps<0,\n    }\n    for code,mask in checks.items():\n        if int(mask.sum()): findings.append({"code":code,"count":int(mask.sum()),"rows":df.index[mask].astype(int).tolist()[:20]})\n    stale=0\n    for _,g in df.sort_values([\'instrument_id\',\'date\']).groupby(\'instrument_id\'):\n        stale += int((g.close.groupby((g.close!=g.close.shift()).cumsum()).transform(\'size\')>=8).sum())\n    if stale: findings.append({"code":"STALE_CLOSE_RUN","count":stale})\n    return {"valid":not findings,"findings":findings,"row_count":len(df),"instrument_count":int(df.instrument_id.nunique()),"date_count":int(df.date.nunique())}\n\ndef validate_database(db_path: str | Path) -> dict:\n    con=sqlite3.connect(db_path)\n    try:\n        integrity=con.execute(\'PRAGMA integrity_check\').fetchone()[0]\n        fk=con.execute(\'PRAGMA foreign_key_check\').fetchall()\n        prices=pd.read_sql_query(\'SELECT * FROM prices_daily\',con)\n        result=validate_prices(prices); result.update({"integrity":integrity,"foreign_key_violations":len(fk)})\n        result[\'valid\']=result[\'valid\'] and integrity==\'ok\' and not fk\n        return result\n    finally: con.close()\n'
(ROOT/'tools'/'synthetic_equity_generator.py').write_text(GENERATOR_SOURCE)
(ROOT/'tools'/'database_builder.py').write_text(DATABASE_SOURCE)
(ROOT/'tools'/'dataset_validator.py').write_text(VALIDATOR_SOURCE)
sys.path.insert(0,str(ROOT/'tools'))
from synthetic_equity_generator import SyntheticConfig,generate_synthetic_equity_market,save_datasets
from database_builder import build_database
from dataset_validator import validate_prices,validate_database


## 3. Generate the clean canonical market

The generator combines a persistent four-state regime process, a market factor, sector shocks, heterogeneous betas, and idiosyncratic risk. It derives consistent OHLC observations, state-sensitive volume and spreads, a trading calendar, corporate-action events, a clean panel, a separately corrupted panel, a defect log, and a hashed manifest. The fixed seed makes this reference world reproducible.


In [ ]:
config=SyntheticConfig(seed=42,n_assets=30,n_days=756)
bundle=generate_synthetic_equity_market(config)
paths=save_datasets(bundle,ROOT/'datasets')
bundle['manifest'], bundle['instruments'].head(), bundle['prices'].head()


## 4. Validate clean data and challenge the validator

The clean panel must pass every control. The separately corrupted panel must fail and identify the injected defect classes. Both assertions are executable stopping conditions. This dual test distinguishes a meaningful validator from a permissive function that reports success whenever it receives a familiar table.


In [ ]:
clean_report=validate_prices(bundle['prices'])
defect_report=validate_prices(bundle['defective_prices'])
assert clean_report['valid']
assert not defect_report['valid']
print('Clean:',clean_report)
print('Defect codes:',[x['code'] for x in defect_report['findings']])


## 5. Build the canonical SQLite database

The database encodes identity and financial invariants through primary keys, foreign keys, positivity rules, non-negative volume and spreads, and OHLC consistency. Manifests and provenance are stored beside the observations. The build is accepted only when SQLite integrity, foreign-key checks, and the validator all succeed.


In [ ]:
db_result=build_database(bundle,ROOT/'database'/'synthetic_equity_market.db')
db_validation=validate_database(ROOT/'database'/'synthetic_equity_market.db')
assert db_result['integrity']=='ok' and db_validation['valid']
db_result,db_validation


## 6. Inspect regimes, sectors, and coverage

This diagnostic is not decorative. It confirms that all six sectors are represented, all intended regimes occur, the date range is complete, and the price-panel row count reconciles to assets multiplied by dates. Later models will therefore face calm, directional, mean-reverting, and crisis conditions within one controlled environment.


In [ ]:
import pandas as pd
regime_counts=bundle['regimes'].regime.value_counts()
sector_counts=bundle['instruments'].sector.value_counts()
summary={'regimes':regime_counts.to_dict(),'sectors':sector_counts.to_dict(),'rows':len(bundle['prices']),'date_range':[bundle['manifest']['start'],bundle['manifest']['end']]}
summary


## 7. Register the reusable data tools

The notebook closes the Notebook → Tool loop by writing a ToolSpec for each capability and recording the version, validation status, and implementation hash in a registry. Future notebooks can consume these tools without reimplementing the market or expanding their permissions. Registration preserves evidence; it does not confer live authority.


In [ ]:
TOOL_SPECS=[{'schema_version': '1.0.0', 'tool_id': 'synthetic_equity_generator', 'name': 'Synthetic Equity Generator', 'version': '1.0.0', 'purpose': 'Generate deterministic synthetic equity markets with sectors, regimes, corporate actions, and a separate defect layer.', 'inputs': {'config': 'SyntheticConfig'}, 'outputs': {'bundle': 'tables and manifest'}, 'errors': [{'code': 'INVALID_INPUT', 'meaning': 'Input violates the canonical contract.', 'recovery': 'Correct the reported schema or quality finding.'}], 'permissions': [{'capability': 'read', 'effect': 'allow', 'scope': 'local synthetic datasets'}, {'capability': 'write', 'effect': 'allow', 'scope': 'designated local artifact directory'}, {'capability': 'execute', 'effect': 'deny', 'scope': 'brokerage, live markets, and external trading systems'}], 'side_effects': ['Writes local synthetic files when explicitly invoked.'], 'provenance': {'source_notebook': 'NB00_Synthetic_Equity_Market_Engine.ipynb', 'build_action': 'A059-A082', 'content_hash_required': True}, 'tests': [{'test_id': 'synthetic_equity_generator_test_1', 'kind': 'unit', 'expected': 'same seed produces identical hashes'}, {'test_id': 'synthetic_equity_generator_test_2', 'kind': 'unit', 'expected': 'regime and sector coverage'}], 'status': 'validated'}, {'schema_version': '1.0.0', 'tool_id': 'canonical_database_builder', 'name': 'Canonical Database Builder', 'version': '1.0.0', 'purpose': 'Build a constrained SQLite research database from a validated synthetic equity bundle.', 'inputs': {'bundle': 'validated dataset bundle', 'db_path': 'path'}, 'outputs': {'integrity': 'status', 'counts': 'table row counts'}, 'errors': [{'code': 'INVALID_INPUT', 'meaning': 'Input violates the canonical contract.', 'recovery': 'Correct the reported schema or quality finding.'}], 'permissions': [{'capability': 'read', 'effect': 'allow', 'scope': 'local synthetic datasets'}, {'capability': 'write', 'effect': 'allow', 'scope': 'designated local artifact directory'}, {'capability': 'execute', 'effect': 'deny', 'scope': 'brokerage, live markets, and external trading systems'}], 'side_effects': ['Writes local synthetic files when explicitly invoked.'], 'provenance': {'source_notebook': 'NB00_Synthetic_Equity_Market_Engine.ipynb', 'build_action': 'A059-A082', 'content_hash_required': True}, 'tests': [{'test_id': 'canonical_database_builder_test_1', 'kind': 'unit', 'expected': 'SQLite integrity check'}, {'test_id': 'canonical_database_builder_test_2', 'kind': 'unit', 'expected': 'foreign-key enforcement'}, {'test_id': 'canonical_database_builder_test_3', 'kind': 'unit', 'expected': 'row-count reconciliation'}], 'status': 'validated'}, {'schema_version': '1.0.0', 'tool_id': 'dataset_validator', 'name': 'Dataset Validator', 'version': '1.0.0', 'purpose': 'Detect schema, null, duplicate, price, OHLC, volume, spread, and stale-price defects.', 'inputs': {'prices': 'dataframe or canonical database'}, 'outputs': {'valid': 'boolean', 'findings': 'standardized list'}, 'errors': [{'code': 'INVALID_INPUT', 'meaning': 'Input violates the canonical contract.', 'recovery': 'Correct the reported schema or quality finding.'}], 'permissions': [{'capability': 'read', 'effect': 'allow', 'scope': 'local synthetic datasets'}, {'capability': 'write', 'effect': 'allow', 'scope': 'designated local artifact directory'}, {'capability': 'execute', 'effect': 'deny', 'scope': 'brokerage, live markets, and external trading systems'}], 'side_effects': ['Writes local synthetic files when explicitly invoked.'], 'provenance': {'source_notebook': 'NB00_Synthetic_Equity_Market_Engine.ipynb', 'build_action': 'A059-A082', 'content_hash_required': True}, 'tests': [{'test_id': 'dataset_validator_test_1', 'kind': 'unit', 'expected': 'clean dataset passes'}, {'test_id': 'dataset_validator_test_2', 'kind': 'unit', 'expected': 'controlled defects are detected'}], 'status': 'validated'}]
import hashlib
registry=[]
implementation={'synthetic_equity_generator':'synthetic_equity_generator.py','canonical_database_builder':'database_builder.py','dataset_validator':'dataset_validator.py'}
for spec in TOOL_SPECS:
    (ROOT/'tools'/f"{spec['tool_id']}.tool.json").write_text(json.dumps(spec,indent=2))
    p=ROOT/'tools'/implementation[spec['tool_id']]
    registry.append({'artifact_id':spec['tool_id'],'version':'1.0.0','status':'validated','content_hash':hashlib.sha256(p.read_bytes()).hexdigest()})
(ROOT/'registry'/'tool_registry_day2.json').write_text(json.dumps(registry,indent=2))
registry


## Conclusion: a governed market world is ready for intelligence

NB00 successfully constructs the canonical synthetic-equity environment required by the course. With seed 42, the validated reference execution creates thirty securities, allocates five firms to each of six sectors, and generates 756 business days from 2 January 2023 through 24 November 2025. The resulting daily price table contains 22,680 instrument-date observations. The regime history includes 418 calm days, 159 trend days, 147 mean-reverting days, and 32 crisis days, giving later models a controlled mixture of persistence, reversal, ordinary volatility, and stressed liquidity rather than a single homogeneous process.

The clean panel passes with no findings. The negative-control panel triggers the intended checks for required-field nulls, duplicate keys, inconsistent OHLC relationships, negative volume, and stale prices. That result is central: the notebook shows not only that acceptable data can proceed, but also that known bad data are prevented from quietly entering the research chain. The canonical SQLite database then passes its integrity and foreign-key checks. Its reconciled contents include thirty instruments, 756 calendar observations, 22,680 price rows, 92 corporate-action records, five documented defects, one dataset manifest, and four provenance records.

Three validated tools emerge from the notebook: the Synthetic Equity Generator, the Canonical Database Builder, and the Dataset Validator. Their implementations are hashed and registered with explicit inputs, outputs, errors, tests, side effects, and permissions. Each may read, calculate, and write designated local research artifacts, but none may connect to a broker, consume live execution credentials, place an order, or allocate capital. The notebook therefore increases the system’s capability while preserving the authority boundary defined by NB00A.

These results do not show that a trading strategy is profitable. Synthetic prices are teaching instruments, corporate actions are not fully reflected in adjusted histories, and performance in a designed world cannot establish investability. What NB00 establishes is more precise: downstream research can now be reproduced against a known data-generating process, validators can be tested against known defects, database invariants can be enforced, and every material table can be connected to a manifest and content hash.

The immediate concatenation point is NB00B. That notebook should consume the registered database rather than regenerate prices, verify the dataset identity and hash, engineer features and forward labels, create chronological training, validation, and test partitions, fit the deliberately simple KNN baseline, convert predictions into bounded positions, apply costs, and generate stress and audit evidence. NB01 can then replace the single baseline with a comparative model–strategy–portfolio laboratory while preserving the same data and control contracts.

Later stages will promote the three data capabilities into the Tool Factory, compose validated tools into governed skills, and make those skills available to bounded research, risk, and audit agents. Because NB00 preserves provenance and controlled failure behavior, those agents can share a common world without silently changing its history. The model may eventually propose, governance will challenge, audit will evaluate, and the human will remain accountable. NB00’s achievement is thus foundational: before building intelligence, knowledge, organization, or autonomy, the course has built a market world that can be inspected, reproduced, challenged, and trusted for the limited purpose it declares.
